# Track 3 · 03 Muse prepare — the sealed-phase training data onto Drive

**One job.** Put NEMAR `nm000287`, the official Muse training set, on Drive as windows,
targets and splits, in the same layout `track3_01` writes for Sleep-EDF.

Self-contained: upload this one file. Not needed for the warm-up.

---

## The data

From the deposit's README (`nm000287` v1.0.0): 540 at-home recordings from 203 people,
Muse S headband, four channels **TP9, AF7, AF8, TP10 at 128 Hz**, 157.5 hours, about
1.2 GB. Each recording carries one `n2_onset` event; there is no full hypnogram. The
recordings are short (6–30 min) and start near bedtime: N2 onset has a median of 690 s.

**Licence: CC-BY-NC-SA-4.0.** Non-commercial, share-alike. Fine for the competition;
flag before any commercial use.

## Windows, targets, preprocessing: the Sleep-EDF rules, assumed

The organisers say the sealed data gets the same preprocessing as training, but the
Muse task config is not published yet. Until it is, this applies the `sleep_onset` rules
from `track3_01` unchanged: 5 s windows every 5 s over the 1,200 s before N2 onset,
target `clip(n2_onset - window_stop, 0, 600)`, and neuralbench's default pipeline. At
128 Hz that pipeline does more than on Sleep-EDF:

| step | value | at 128 Hz |
|---|---|---|
| notch | 50 and 60 Hz plus harmonics below Nyquist | **50 and 60 Hz both apply** (Nyquist 64) |
| band-pass | 0.1 – 75 Hz | 75 Hz is above Nyquist, so only the 0.1 Hz highpass |
| resample | 120 Hz | 600 samples per window |
| scale | sklearn `RobustScaler` per channel over the whole recording | |
| clamp | ±20 | |

**Known caveat.** Every recording ends exactly 300 s after onset, and the scaler is fit on
the whole recording, so it sees samples from the future. That is the harness's default
behaviour, but the deposit's README says preprocessing must not use future samples, and
the new streaming task may change it. Rerun this notebook with the published rule once
it is out.

## Splits: both groups the sealed score averages

The sealed W-bMAE is macro-averaged over **seen** sleepers (new nights from people in the
training data) and **unseen** sleepers. Our split covers both, seed 33:

| group | what | used for |
|---|---|---|
| `test_seen` | the deposit's own 40 test recordings, 40 people who also have training nights | seen score |
| `test_unseen` | every recording of 20% of the 163 people with no supplied test night | unseen score |
| `val` | every recording of 20% of the remaining people | choosing epochs |
| `train` | everything else, including the seen-test people's other nights | training |

This is our split, not the organisers'. The sealed unseen cohort is not in the deposit.

## Cost

About 1.2 GB downloaded to temporary disk and deleted per recording, about 380 MB
written to Drive (windows stored as float16; rounding error under 0.005 against a
signal sd of about 1). A few minutes. Resumes on rerun.

## Setup

In [ ]:
ROOT = '/content/drive/MyDrive/Neurips26'
WORKERS = 8          # recordings downloaded and processed in parallel (each is small)

# Assumed equal to the Sleep-EDF task until the Muse config is published.
SFREQ = 120.0
NOTCH = (50.0, 60.0)
BANDPASS = (0.1, 75.0)
CLAMP = 20.0
DURATION = 5.0
STRIDE = 5.0
MAX_PRE_N2_S = 1200.0
CAP_S = 600.0
BIN_EDGES = (0.0, 40.0, 90.0, 300.0, 600.0)

DEPOSIT = 'https://data.nemar.org/nm000287/v1.0.0'
SPLIT_SEED = 33
UNSEEN_FRAC = 0.2
VAL_FRAC = 0.2

print('NOTEBOOK VERSION: track3_03 v1')

In [ ]:
%pip install -q mne

import importlib
for mod in ('mne', 'sklearn', 'numpy'):
    print(f'{mod:8} {importlib.import_module(mod).__version__}')

In [ ]:
import csv, io, json, math, shutil, time, urllib.request, warnings
warnings.filterwarnings('ignore')
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import numpy as np, mne
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
mne.set_log_level('ERROR')

if ROOT.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

OUT = Path(ROOT) / 'Track3' / 'muse'
WIN = OUT / 'windows'
WIN.mkdir(parents=True, exist_ok=True)
TMP = Path('/content/muse_tmp') if Path('/content').exists() else OUT.parent / '_muse_tmp'
TMP.mkdir(parents=True, exist_ok=True)


def get(rel, timeout=120):
    for attempt in range(4):
        try:
            with urllib.request.urlopen(f'{DEPOSIT}/{rel}', timeout=timeout) as r:
                return r.read()
        except Exception as e:
            if attempt == 3:
                raise RuntimeError(f'{rel}: {e}')
            time.sleep(5 * (attempt + 1))


def tsv(rel):
    return list(csv.DictReader(io.StringIO(get(rel).decode('utf-8-sig')), delimiter='\t'))


people = [r['participant_id'] for r in tsv('participants.tsv')]
with ThreadPoolExecutor(16) as pool:
    tables = dict(zip(people, pool.map(lambda p: tsv(f'{p}/{p}_sessions.tsv'), people)))
recs = [dict(person=p, session=r['session_id'], supplied_split=r['split'],
             n2_onset=float(r['n2_onset']))
        for p in people for r in tables[p]]
print(f'{len(recs)} recordings from {len(people)} people '
      f'(the deposit says 540 from 203)')

## Step 1 — download, preprocess and window each recording

The onset comes from each recording's `events.tsv`, the one `n2_onset` event, and is
checked against the session table. The signal is read with MNE's BrainVision reader,
which applies the header's 0.1 µV resolution.

In [ ]:
def muse_windows(vhdr, t_onset):
    """One recording -> (X, y, starts, info), with Track 3's window rule and the
    harness's default preprocessing, in neuralset's order."""
    raw = mne.io.read_raw_brainvision(vhdr, preload=True)
    raw.pick('eeg')
    rec_stop = raw.n_times / raw.info['sfreq']
    info = dict(native_sfreq=float(raw.info['sfreq']), rec_s=rec_stop, n2_onset=t_onset,
                channels=list(raw.ch_names))
    t0 = max(0.0, t_onset - MAX_PRE_N2_S)
    t_stop = min(t_onset, rec_stop)
    span = (t_stop - DURATION) - t0
    if t_stop <= t0 or span < -1e-10:
        return None, None, None, {**info, 'skipped': 'pre-N2 region shorter than a window'}
    n = math.floor(span / STRIDE + 1e-9) + 1
    starts = t0 + np.arange(n) * STRIDE

    nyq = raw.info['sfreq'] / 2
    freqs = [f for base in NOTCH for f in np.arange(base, min(nyq, 301), base)]
    if freqs:                      # at 128 Hz: 50 and 60
        raw.notch_filter(freqs, phase='zero', filter_length='auto', verbose=False)
    lo, hi = BANDPASS
    raw.filter(lo, hi if hi < nyq else None, verbose=False)
    raw.resample(SFREQ, verbose=False)
    d = RobustScaler().fit_transform(raw.get_data().T).T

    n_times = int(round(DURATION * SFREQ))
    idx = np.round(starts * SFREQ).astype(int)
    X = np.clip(np.stack([d[:, i:i + n_times] for i in idx]), -CLAMP, CLAMP)
    y = np.clip(t_onset - (starts + DURATION), 0.0, CAP_S).astype('float32')
    return X.astype('float16'), y, starts, {**info, 'n_windows': int(n),
                                             'notch_hz': [float(f) for f in freqs]}


def do_rec(r):
    name = f'{r["person"]}_{r["session"]}'
    dest = WIN / f'{name}.npz'
    if dest.exists():
        return name, 'kept', json.loads(str(np.load(dest)['info']))
    rel = f'{r["person"]}/{r["session"]}/eeg/{name}_task-sleeponset'
    folder = TMP / name
    folder.mkdir(exist_ok=True)
    try:
        for ext in ('_eeg.vhdr', '_eeg.vmrk', '_eeg.eeg'):
            (folder / f'{name}_task-sleeponset{ext}').write_bytes(get(rel + ext))
        ev = [e for e in tsv(rel + '_events.tsv') if e['trial_type'] == 'n2_onset']
        if len(ev) != 1:
            raise RuntimeError(f'expected one n2_onset event, found {len(ev)}')
        t_onset = float(ev[0]['onset'])
        if abs(t_onset - r['n2_onset']) > 1e-3:
            raise RuntimeError(f'events.tsv onset {t_onset} != session table {r["n2_onset"]}')
        X, y, starts, info = muse_windows(folder / f'{name}_task-sleeponset_eeg.vhdr', t_onset)
    finally:
        shutil.rmtree(folder, ignore_errors=True)
    info.update(person=r['person'], session=r['session'], supplied_split=r['supplied_split'])
    if X is None:
        np.savez(dest, info=json.dumps(info))
        return name, 'no windows', info
    np.savez(dest, X=X, y=y, starts=starts, info=json.dumps(info))
    return name, 'new', info


t0 = time.time()
done, failed = {}, {}
with ThreadPoolExecutor(WORKERS) as pool:
    futs = {pool.submit(do_rec, r): r for r in recs}
    for i, f in enumerate(as_completed(futs), 1):
        r = futs[f]
        try:
            name, status, info = f.result()
            done[name] = info
            if i % 50 == 0 or i == len(recs):
                print(f'  {i:3d}/{len(recs)}  {name} {status}  {time.time() - t0:.0f}s')
        except Exception as e:
            failed[f'{r["person"]}_{r["session"]}'] = repr(e)
            print(f'  FAILED {r["person"]} {r["session"]}: {e}')
print(f'\n{len(done)} recordings processed, {len(failed)} failed, {time.time() - t0:.0f}s')
if failed:
    print('Rerun this cell: finished recordings are kept and only the failures are retried.')

## Step 2 — splits, manifest and checks

In [ ]:
seen_people = sorted({r['person'] for r in recs if r['supplied_split'] == 'test'})
pool_people = sorted(set(people) - set(seen_people))
rest, unseen = train_test_split(pool_people, test_size=UNSEEN_FRAC,
                                random_state=SPLIT_SEED, shuffle=True)
_, val = train_test_split(sorted(rest), test_size=VAL_FRAC,
                          random_state=SPLIT_SEED, shuffle=True)
unseen, val = set(unseen), set(val)


def group_of(person, supplied):
    if person in unseen:
        return 'test_unseen'
    if person in val:
        return 'val'
    return 'test_seen' if supplied == 'test' else 'train'


def bin_counts(y):
    b = np.digitize(y, BIN_EDGES[1:-1])
    return [int((b == k).sum()) for k in range(len(BIN_EDGES) - 1)]


GROUPS = ('train', 'val', 'test_seen', 'test_unseen')
manifest_recs, ys, problems = {}, {g: [] for g in GROUPS}, []
for f in sorted(WIN.glob('sub-*.npz')):
    z = np.load(f)
    info = json.loads(str(z['info']))
    g = group_of(info['person'], info['supplied_split'])
    entry = dict(info, group=g, file=f.name)
    if 'X' in z:
        X, y, st = z['X'], z['y'], z['starts']
        entry['n_windows'] = len(y)
        ys[g].append(y)
        if X.shape[1:] != (4, int(DURATION * SFREQ)):
            problems.append(f'{f.name}: shape {X.shape}')
        if info['channels'] != ['TP9', 'AF7', 'AF8', 'TP10']:
            problems.append(f'{f.name}: channels {info["channels"]}')
        if len(st) > 1 and not np.allclose(np.diff(st), STRIDE):
            problems.append(f'{f.name}: window starts not {STRIDE} s apart')
        if not (np.isfinite(X.astype('float32')).all() and np.abs(X).max() <= CLAMP):
            problems.append(f'{f.name}: values out of range')
        if y[-1] != 0.0:
            problems.append(f'{f.name}: last window target {y[-1]}, expected 0')
    manifest_recs[f.stem] = entry

per_group = {}
for g in GROUPS:
    y = np.concatenate(ys[g]) if ys[g] else np.zeros(0)
    per_group[g] = dict(
        people=len({r['person'] for r in manifest_recs.values() if r['group'] == g}),
        recordings=sum(1 for r in manifest_recs.values() if r['group'] == g and r.get('n_windows')),
        windows=int(len(y)), per_bin=bin_counts(y))
    print(f'  {g:11}  {per_group[g]["people"]:3d} people  {per_group[g]["recordings"]:3d} recordings  '
          f'{len(y):6,d} windows   per bin {per_group[g]["per_bin"]}')

skipped = {k: r['skipped'] for k, r in manifest_recs.items() if 'skipped' in r}
leak = {r['person'] for r in manifest_recs.values() if r['group'] in ('val', 'test_unseen')} & \
       {r['person'] for r in manifest_recs.values() if r['group'] == 'train'}
if leak:
    problems.append(f'held-out people also in train: {sorted(leak)[:5]}')
print(f'\n{len(manifest_recs)} recordings on Drive, {len(skipped)} without windows: '
      f'{skipped or "none"}')
print('checks:', 'all passed' if not problems else problems[:10])

(OUT / 'manifest.json').write_text(json.dumps(dict(
    source='NEMAR nm000287 v1.0.0, Muse sleep-onset EEG (EEG/EMG Foundation Challenge 2026, Track 03)',
    licence='CC-BY-NC-SA-4.0', deposit=DEPOSIT,
    preprocessing=dict(sfreq=SFREQ, notch=NOTCH, bandpass=BANDPASS, scaler='RobustScaler',
                       baseline=None, clamp=CLAMP, stored_dtype='float16',
                       note='Sleep-EDF task defaults assumed; scaler fit on the whole recording'),
    windows=dict(duration=DURATION, stride=STRIDE, max_pre_n2_s=MAX_PRE_N2_S, cap_s=CAP_S,
                 target='clip(n2_onset - window_stop, 0, cap_s)'),
    bin_edges=BIN_EDGES,
    split=dict(seed=SPLIT_SEED, unseen_frac=UNSEEN_FRAC, val_frac=VAL_FRAC,
               test_seen='supplied test recordings', groups=GROUPS),
    per_group=per_group, recordings=manifest_recs, failed=failed), indent=1))
size = sum(f.stat().st_size for f in WIN.glob('*.npz')) / 1e6
print(f'\nmanifest -> {OUT / "manifest.json"}   windows on Drive: {size:.0f} MB')
ready = len(recs) == 540 and not failed and not problems
print('READY' if ready else 'NOT READY: see the counts and checks above')